# Geometry predictors linked to six finalized hydrograph responses

14 October 2026. **Peak-time shift is the leading target; peak-excess change is the second primary target.** Centroid, duration, volume and asymmetry help distinguish translation, spreading and redistribution. This notebook uses the completed reruns and fixed-baseline **0.5% shared window**, with 1% and 2% sensitivity.

This continues the earlier predictor investigation without replacing its notebook or the user-edited metric examples. The old five-network response tables and hydraulic predictors are not imported. Physical geometry is recalculated from the parameter files whose hashes were recorded in the completed-run plan. No routing, calibration or response definitions were changed.

The results below are **exploratory fixed-panel benchmarks**, not a final selected predictor model. Broad candidate associations are for review; they did not select the fitted columns.

**Current results:** see [Resistance and unit–compound comparison](hydrograph_predictor_scope_comparison.ipynb). That notebook incorporates the existing domain sources and resistance, with identical-cohort comparisons. The results retained below are the earlier restricted-domain benchmark.

**Follow-up correction: existing domain sources (14 October 2026).** The 603 `unavailable` records below are failures of this investigation's domain loader, **not 603 unclassified domains**. Of these, 295 already have explicit stable parent–child anchor mappings in their resistance comparison tables; the loader incorrectly required equal numeric node IDs. Those comprise 208 containing-compound contexts, 69 closed units and 18 closed compound units. Another 269 are island states: all have recorded source-unit/anchor metadata and analysis directories; 247 also have readily located planning contracts (223 containing-compound contexts and 24 closed units). The remaining 22 contracts require further path/source tracing. The other 39 failures concern current endpoint/link agreement.

All 15 parent examples already have slope ownership and domain-audit tables. Reuse the established domain selector and anchor mapping; no new domain classification scheme is needed. The model results below remain the original restricted-cohort benchmark and have **not** been recomputed with these sources integrated. Do not interpret their local-domain coverage as the availability of domain definitions in the project.

## Notation: metric × scope × state

Use $U$ for the edited unit, $C$ for its containing compound, and $N$ for the whole network; subscripts $p,c$ denote parent and child. **The current preliminary tables use $D$ for the matched domain actually loaded.** It can be $U$ or $C$; `domain_type` records which. We must separate those scopes before comparing local and compound predictors. These display changes do not repair domain loading or refit the preliminary models.

$$R_X^D=\frac{X_c^D}{X_p^D},\qquad \Delta X^D=X_c^D-X_p^D.$$

| Metric | Symbol |
|---|---|
| Mean route length | $\mu_L$ |
| Route-length SD and CV | $\sigma_L$, $\mathrm{CV}_L=\sigma_L/\mu_L$ |
| Length-weighted mean width and width CV | $\bar W$, $\mathrm{CV}_W$ |
| Mean split dominance | $\bar d$ |
| Expected bifurcations encountered | $\mathbb E[N_b]$ |
| Route entropy | $H_{\mathrm{route}}$ |
| Mean inverse-Simpson effective branch count | $\bar N_{\mathrm{eff},2}$ |
| Mean split impact | $\bar I$ |
| Entry exposure | $r_{\mathrm{in}}$ |
| Equivalent resistance | $K_{\mathrm{eq}}$ |

$\bar N_{\mathrm{eff},2}$ uses $1/\sum_jp_j^2$; it is distinct from the entropic effective count $e^{H_b}$.

The additional combined expressions remain visible:

$$r_{\mathrm{in},p}^{D}\frac{\Delta\mu_L^D}{\mu_{L,p}^{N}},\qquad
r_{\mathrm{in},p}^{D}\frac{\Delta\sigma_L^D}{\mu_{L,p}^{N}},\qquad
\frac{\mu_{L,p}^{D}}{\mu_{L,p}^{N}}.$$

These are preliminary exposure/scale hypotheses, not new basic geometric metrics. The immediate plan is to reuse existing domain mappings, distinguish $U$ from $C$, and review individual metrics before selecting combined predictors. The worked-example notebook remains the metric dictionary.


In [12]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'RAPID/src/rapid_tools').is_dir())
OUT=ROOT/'RAPID/outputs/hydrograph_predictors_20261014'
sys.path.insert(0,str(ROOT/'RAPID/src'))
from rapid_tools.hydrograph_response import load, METRICS, threshold_figure
pio.renderers.default='plotly_mimetype'
states,responses=load('0.5%')
d=pd.read_parquet(OUT/'linked_responses.parquet')
manifest=json.loads((OUT/'manifest.json').read_text())
design=json.loads((OUT/'validation_design.json').read_text())
scores=pd.read_csv(OUT/'validation_scores.csv')
associations=pd.read_csv(OUT/'individual_associations.csv',dtype={'example':str})
audit=pd.read_csv(OUT/'local_domain_audit.csv',dtype={'example':str})
assert not d.duplicated(['example','state']).any()
assert len(d)==len(responses)==2187
display(pd.Series({k:manifest[k] for k in ['comparisons','examples','matched','unmatched','historical_unmatched_aliases']}))
# Display notation only; source data and fitted predictor columns are unchanged.
from IPython.display import Markdown
METRIC_SYMBOLS={
 'route_length_mean':r'\mu_L','route_length_sd':r'\sigma_L','route_length_cv':r'\mathrm{CV}_L',
 'width_mean':r'\bar W','width_cv':r'\mathrm{CV}_W','dominance_mean':r'\bar d',
 'expected_splits':r'\mathbb{E}[N_b]','route_entropy':r'H_{\mathrm{route}}',
 'effective_branches_mean':r'\bar N_{\mathrm{eff},2}','split_impact_mean':r'\bar I',
}
RESPONSE_SYMBOLS={
 'peak_shift_hours':r'\Delta t_{\mathrm{peak}}\;[\mathrm{h}]',
 'peak_excess_change_pct':r'100(R_{E_{\max}}-1)\;[\%]',
 'centroid_shift_hours':r'\Delta\mu_t\;[\mathrm{h}]',
 'duration80_change_pct':r'100(R_{D_{80}}-1)\;[\%]',
 'volume_change_pct':r'100(R_V-1)\;[\%]',
 'asymmetry_change':r'\Delta A',
}
EXTRA_SYMBOLS={
 'domain_entry_exposure':r'r_{\mathrm{in},p}^{D}',
 'domain_parent_length_over_network':r'\mu_{L,p}^{D}/\mu_{L,p}^{N}',
 'exposed_length_change':r'r_{\mathrm{in},p}^{D}\,\Delta\mu_L^{D}/\mu_{L,p}^{N}',
 'exposed_route_sd_change':r'r_{\mathrm{in},p}^{D}\,\Delta\sigma_L^{D}/\mu_{L,p}^{N}',
}
def symbol(name):
    if not isinstance(name,str):return name
    if name in RESPONSE_SYMBOLS:return '$'+RESPONSE_SYMBOLS[name]+'$'
    if name in EXTRA_SYMBOLS:return '$'+EXTRA_SYMBOLS[name]+'$'
    for prefix,scope in [('network_','N'),('domain_','D')]:
        if name.startswith(prefix):
            rest=name[len(prefix):]
            for operation in ['parent','child','ratio','delta']:
                if rest.startswith(operation+'_'):
                    metric=rest[len(operation)+1:]
                    if metric not in METRIC_SYMBOLS:return name
                    x=METRIC_SYMBOLS[metric]
                    value={'parent':rf'({x})_p^{{{scope}}}', 'child':rf'({x})_c^{{{scope}}}',
                           'ratio':rf'R_{{{x}}}^{{{scope}}}', 'delta':rf'\Delta({x})^{{{scope}}}'}[operation]
                    return '$'+value+'$'
    return name

def show_symbols(obj):
    if not isinstance(obj,(pd.DataFrame,pd.Series)):
        display(obj);return
    table=obj.to_frame() if isinstance(obj,pd.Series) else obj.copy()
    table=table.rename(columns=symbol,index=symbol)
    for col in table.columns:
        if table[col].dtype==object:table[col]=table[col].map(symbol)
    def clean(x):
        return str(x).replace('|',r'\|').replace('\n',' ')
    header=['']+[clean(x) for x in table.columns]
    lines=['| '+' | '.join(header)+' |','| '+' | '.join(['---']*len(header))+' |']
    for idx,row in table.iterrows():lines.append('| '+' | '.join([clean(idx),*[clean(x) for x in row]])+' |')
    display(Markdown('\n'.join(lines)))


comparisons                     2187
examples                          15
matched                         2187
unmatched                          0
historical_unmatched_aliases       0
dtype: int64

## 1. The six responses remain fixed

For each comparison, use one scalar per response. Peak time is the **first saved global maximum in the common analysis window**; peak excess is that maximum minus the shared baseline. Neither target changes to a matched-crest definition in example 54.

Signed changes are child minus parent. Percentages are $100(X_c/X_p-1)$. Timing units are hours; percentage errors are percentage points. No interpolation of a different peak time is introduced.

The baseline is the fixed minimum across each example's completed states after input start and initial/final equilibria. The window starts at the earliest sustained departure across states and ends at the latest return to final equilibrium within the chosen tolerance. Endpoint excess is retained. Distribution metrics describe this finite window, including the transition to a higher final plateau in 54.

In [13]:
response_contract=pd.DataFrame([
 ('Timing and duration','peak_shift_hours','PRIMARY: time of maximum discharge','Length change, width change, resistance; crest dominance can also matter'),
 ('Timing and duration','centroid_shift_hours','Supporting: excess-volume centre','Allocation-weighted distance; exposure; retained late volume'),
 ('Timing and duration','duration80_change_pct','Supporting: central 80% volume duration','Route-length variability, branch balance and repeated splitting'),
 ('Magnitude','peak_excess_change_pct','PRIMARY: maximum above shared baseline','Width/resistance changes, route variability, allocation balance'),
 ('Magnitude','volume_change_pct','Supporting: finite-window excess volume','Check truncation/equilibrium transition before interpreting geometry'),
 ('Temporal shape','asymmetry_change','Supporting: quantile asymmetry','Unequal routes and balance; finite-window effects'),
],columns=['group','response','role','candidate physical connections — hypotheses'])
show_symbols(response_contract)
show_symbols(d[list(METRICS)].describe().T[['count','min','50%','max']])

|  | group | response | role | candidate physical connections — hypotheses |
| --- | --- | --- | --- | --- |
| 0 | Timing and duration | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | PRIMARY: time of maximum discharge | Length change, width change, resistance; crest dominance can also matter |
| 1 | Timing and duration | $\Delta\mu_t\;[\mathrm{h}]$ | Supporting: excess-volume centre | Allocation-weighted distance; exposure; retained late volume |
| 2 | Timing and duration | $100(R_{D_{80}}-1)\;[\%]$ | Supporting: central 80% volume duration | Route-length variability, branch balance and repeated splitting |
| 3 | Magnitude | $100(R_{E_{\max}}-1)\;[\%]$ | PRIMARY: maximum above shared baseline | Width/resistance changes, route variability, allocation balance |
| 4 | Magnitude | $100(R_V-1)\;[\%]$ | Supporting: finite-window excess volume | Check truncation/equilibrium transition before interpreting geometry |
| 5 | Temporal shape | $\Delta A$ | Supporting: quantile asymmetry | Unequal routes and balance; finite-window effects |

|  | count | min | 50% | max |
| --- | --- | --- | --- | --- |
| $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 2187.0 | -45.68333333333333 | -0.03333333333333333 | 2.1166666666666667 |
| $100(R_{E_{\max}}-1)\;[\%]$ | 2187.0 | -1.3815221820616008 | 0.0025229789285985404 | 33.16930794752513 |
| $\Delta\mu_t\;[\mathrm{h}]$ | 2187.0 | -57.89862479613887 | -0.021287846697701347 | 0.9624409344461229 |
| $100(R_{D_{80}}-1)\;[\%]$ | 2187.0 | -47.98623396561713 | -0.0014354446614572858 | 9.495889318851614 |
| $\Delta A$ | 2187.0 | -0.15287878793917123 | 5.380993454898175e-05 | 0.09515022981885321 |
| $100(R_V-1)\;[\%]$ | 2187.0 | -20.3810397100761 | -0.001558498914355777 | 5.696342819670619 |

## 2. Join and domain audit before interpreting predictors

Canonical keys are **example + state**, resolved through the completed-run plan and response aliases. Historical aliases are cross-checked against canonical fingerprints; aliases never create extra model rows. All 2,187 responses join to current whole-network geometry, including 15 single-centerline comparisons. Historical base-state aliases are labeled `parent_only`, not unmatched children.

Local-domain measurements require matching physical endpoints, a closed single-source/sink DAG, no external crossings through internal nodes, and the same parent/child anchors. Failure is retained in the audit rather than replaced with a guessed local domain.

**Domain type matters:** `closed_unit` and `closed_compound_unit` describe closed unit comparisons; `containing_compound_context` describes the containing compound before/after a member edit, not necessarily the edited footprint. Whole-network summaries are separately prefixed `network_`. Entry exposure uses the whole parent network; allocation inside a local domain is conditional on entering that domain.

In [14]:
show_symbols(pd.crosstab(audit.example,audit.status))
show_symbols(audit.groupby(['status','domain_type'],dropna=False).size().rename('comparisons'))
failures=audit[audit.status.eq('unavailable')].copy()
failures['reason_group']=failures.reason.str.replace(r'No saved local domain membership for .*','No saved local domain membership',regex=True)
show_symbols(failures.reason_group.value_counts())
print('Complete records:',OUT/'response_predictor_join_audit.csv')
print('Unmatched records (empty when successful):',OUT/'unmatched_records.csv')
print('Historical alias audit:',OUT/'historical_alias_join_audit.csv')

|  | unavailable | validated |
| --- | --- | --- |
| 02 | 6 | 30 |
| 03 | 4 | 37 |
| 05 | 1 | 51 |
| 13 | 66 | 1 |
| 15 | 262 | 1 |
| 21 | 4 | 27 |
| 22 | 79 | 330 |
| 37 | 76 | 321 |
| 41 | 63 | 120 |
| 45 | 0 | 94 |
| 46 | 16 | 208 |
| 48 | 1 | 72 |
| 54 | 5 | 77 |
| 55 | 16 | 197 |
| 56 | 4 | 18 |

|  | comparisons |
| --- | --- |
| ('unavailable', nan) | 603 |
| ('validated', 'closed_compound_unit') | 139 |
| ('validated', 'closed_unit') | 352 |
| ('validated', 'containing_compound_context') | 1078 |
| ('validated', 'whole_network_single_centerline') | 15 |

|  | count |
| --- | --- |
| Parent/child domain anchors differ; explicit anchor correspondence is required | 295 |
| No saved local domain membership | 269 |
| Saved domain endpoints differ from current reach endpoints | 38 |
| Domain link IDs are absent from the current physical graph | 1 |

Complete records: /Users/6256481/Code/river-hierarchy/RAPID/outputs/hydrograph_predictors_20261014/response_predictor_join_audit.csv
Unmatched records (empty when successful): /Users/6256481/Code/river-hierarchy/RAPID/outputs/hydrograph_predictors_20261014/unmatched_records.csv
Historical alias audit: /Users/6256481/Code/river-hierarchy/RAPID/outputs/hydrograph_predictors_20261014/historical_alias_join_audit.csv


## 3. Which representations enter this first comparison?

The full descriptive table retains parent and child values, differences and defined ratios. Fitted panels deliberately use a small subset. Positive length/width measures can use ratios; signed or bounded measures use differences. A zero count ratio cannot distinguish 1→0 from 4→0, so expected-split change remains informative. Conditional split means are undefined for a child with no splits; they are stored as NaN, not zero. The fitted local width panel uses **parent** dominance.

The local length panel uses entry-exposed mean-length change normalized by parent network mean route length, parent domain/network length ratio, similarly exposed route-SD change, and entry exposure. Width adds the child/parent mean-width ratio and parent dominance. Topology adds change in expected bifurcations. These are prescribed panels, not winners selected from the association plots.

**Resistance remains an important combined candidate.** It is not imported from older tables because its geometry law/provenance must be consistent. Under wide Chezy and $h=aW^\alpha$, geometry weights are $L/W^{2+3\alpha}$. A new cohort-wide equivalent-resistance calculation, with explicit $\alpha=0.65$ versus $1$ sensitivity and valid domains, is still needed. Harmonic width is not a substitute. The present panels therefore do not settle whether resistance is the better predictor.

P10/P90, junction excess and density remain secondary. Route variance and branch-contrast variance are the same ingredient; route entropy mixes repeated splitting with balance. Dominance, normalized local entropy and effective branch count should first be compared as alternatives. No hydraulic celerity, depth, Manning n, beta, transit time, output-crest flag or hydrograph response is used as a predictor.

In [15]:
panel_rows=[]
for scope,key in [('network','network_panels'),('domain','domain_panels')]:
    for panel,features in design[key].items():
        panel_rows.append(dict(scope=scope,panel=panel,predictors='; '.join(symbol(f) for f in features)))
show_symbols(pd.DataFrame(panel_rows))
show_symbols(pd.read_csv(OUT/'redundancy.csv').sort_values('spearman',key=abs,ascending=False).head(20))

|  | scope | panel | predictors |
| --- | --- | --- | --- |
| 0 | network | length | $R_{\mu_L}^{N}$; $(\mu_L)_p^{N}$; $\Delta(\mathrm{CV}_L)^{N}$ |
| 1 | network | length_width | $R_{\mu_L}^{N}$; $(\mu_L)_p^{N}$; $\Delta(\mathrm{CV}_L)^{N}$; $R_{\bar W}^{N}$; $\Delta(\mathrm{CV}_W)^{N}$ |
| 2 | network | length_width_topology | $R_{\mu_L}^{N}$; $(\mu_L)_p^{N}$; $\Delta(\mathrm{CV}_L)^{N}$; $R_{\bar W}^{N}$; $\Delta(\mathrm{CV}_W)^{N}$; $\Delta(\mathbb{E}[N_b])^{N}$ |
| 3 | domain | length_exposure | $r_{\mathrm{in},p}^{D}\,\Delta\mu_L^{D}/\mu_{L,p}^{N}$; $\mu_{L,p}^{D}/\mu_{L,p}^{N}$; $r_{\mathrm{in},p}^{D}\,\Delta\sigma_L^{D}/\mu_{L,p}^{N}$; $r_{\mathrm{in},p}^{D}$ |
| 4 | domain | length_width_exposure | $r_{\mathrm{in},p}^{D}\,\Delta\mu_L^{D}/\mu_{L,p}^{N}$; $\mu_{L,p}^{D}/\mu_{L,p}^{N}$; $r_{\mathrm{in},p}^{D}\,\Delta\sigma_L^{D}/\mu_{L,p}^{N}$; $r_{\mathrm{in},p}^{D}$; $R_{\bar W}^{D}$; $(\bar d)_p^{D}$ |
| 5 | domain | length_width_topology_exposure | $r_{\mathrm{in},p}^{D}\,\Delta\mu_L^{D}/\mu_{L,p}^{N}$; $\mu_{L,p}^{D}/\mu_{L,p}^{N}$; $r_{\mathrm{in},p}^{D}\,\Delta\sigma_L^{D}/\mu_{L,p}^{N}$; $r_{\mathrm{in},p}^{D}$; $R_{\bar W}^{D}$; $(\bar d)_p^{D}$; $\Delta(\mathbb{E}[N_b])^{D}$ |

|  | scope | a | b | spearman |
| --- | --- | --- | --- | --- |
| 2 | domain | $(\bar d)_p^{D}$ | $(\bar N_{\mathrm{eff},2})_p^{D}$ | -0.9882853947730466 |
| 0 | domain | $\mu_{L,p}^{D}/\mu_{L,p}^{N}$ | $(H_{\mathrm{route}})_p^{D}$ | 0.9370654032128743 |
| 1 | domain | $R_{\bar W}^{D}$ | $(H_{\mathrm{route}})_p^{D}$ | -0.8602612743893435 |

## 4. Individual associations with all six responses

Use these to inspect mechanisms, nonlinearities and network dependence—not to claim causal effects or select predictors before reusing the validation score. Pooled rank association can conceal between-example differences. The next table retains each example separately; constants and insufficient data yield NaN. Children sharing a parent are not independent replicates, so no naive significance tests are attached.

In [16]:
SCOPE='domain'  # 'network' for whole-network summaries
TARGET='peak_shift_hours'
a=associations[(associations.scope==SCOPE)&associations.example.eq('pooled')]
matrix=a.pivot(index='feature',columns='response',values='spearman').reindex(columns=list(METRICS))
fig=px.imshow(matrix.rename(index=symbol,columns=symbol),zmin=-1,zmax=1,color_continuous_scale='RdBu_r',aspect='auto',
              labels=dict(color='Spearman'),title=f'{SCOPE}: exploratory pooled associations with six responses')
fig.update_layout(height=570);fig.show()
within=associations[(associations.scope==SCOPE)&associations.response.eq(TARGET)&~associations.example.eq('pooled')]
show_symbols(within.pivot(index='feature',columns='example',values='spearman').round(3))

|  | 02 | 03 | 05 | 21 | 22 | 37 | 41 | 45 | 46 | 48 | 54 | 55 | 56 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| $\Delta(\mathbb{E}[N_b])^{D}$ | 0.492 | 0.435 | 0.257 | 0.279 | 0.451 | 0.275 | 0.6 | 0.513 | 0.378 | 0.59 | 0.386 | 0.618 | 0.649 |
| $r_{\mathrm{in},p}^{D}$ | 0.044 | -0.104 | 0.086 | -0.269 | 0.13 | 0.03 | -0.033 | -0.273 | -0.219 | -0.096 | 0.139 | -0.024 | -0.268 |
| $(\bar d)_p^{D}$ | 0.528 | 0.162 | 0.288 | 0.256 | 0.017 | 0.053 | 0.072 | 0.06 | 0.209 | -0.184 | 0.058 | 0.078 | 0.156 |
| $(\bar N_{\mathrm{eff},2})_p^{D}$ | -0.528 | -0.138 | -0.288 | -0.25 | -0.053 | -0.053 | -0.047 | -0.037 | -0.224 | 0.17 | -0.037 | -0.079 | -0.074 |
| $\mu_{L,p}^{D}/\mu_{L,p}^{N}$ | 0.267 | -0.301 | -0.606 | -0.735 | 0.052 | 0.073 | -0.019 | -0.465 | -0.261 | -0.11 | -0.335 | 0.033 | -0.732 |
| $(H_{\mathrm{route}})_p^{D}$ | 0.288 | -0.228 | -0.331 | -0.068 | 0.313 | 0.078 | 0.089 | -0.359 | -0.099 | -0.005 | -0.247 | 0.158 | -0.555 |
| $(\bar I)_p^{D}$ | -0.355 | -0.036 | -0.224 | -0.183 | -0.306 | -0.076 | -0.174 | 0.046 | -0.183 | 0.173 | 0.154 | -0.266 | 0.16 |
| $R_{\bar W}^{D}$ | -0.603 | -0.433 | -0.073 | -0.17 | -0.36 | -0.32 | -0.613 | -0.397 | -0.264 | -0.541 | -0.302 | -0.632 | -0.071 |
| $r_{\mathrm{in},p}^{D}\,\Delta\mu_L^{D}/\mu_{L,p}^{N}$ | 0.662 | 0.734 | 0.697 | 0.669 | 0.601 | 0.655 | 0.784 | 0.723 | 0.601 | 0.748 | 0.846 | 0.63 | 0.939 |
| $r_{\mathrm{in},p}^{D}\,\Delta\sigma_L^{D}/\mu_{L,p}^{N}$ | 0.466 | 0.171 | 0.446 | 0.104 | 0.346 | 0.304 | 0.521 | 0.377 | 0.291 | 0.369 | 0.397 | 0.482 | 0.16 |

In [17]:
FEATURE='exposed_length_change'
plot=d[d.comparison.eq('partial collapse')&d.status.eq('validated')].copy()
fig=make_subplots(rows=1,cols=2,subplot_titles=['Peak-time shift — primary','Peak-excess change — primary'])
colors=px.colors.qualitative.Dark24
for i,(ex,g) in enumerate(plot.groupby('example')):
    for col,target in enumerate(['peak_shift_hours','peak_excess_change_pct'],start=1):
        fig.add_trace(go.Scatter(x=g[FEATURE],y=g[target],mode='markers',name=ex,legendgroup=ex,showlegend=col==1,
            marker=dict(size=5,color=colors[i%len(colors)],opacity=.65),customdata=g[['state','domain_type']].to_numpy(),
            hovertemplate='%{customdata[0]}<br>%{customdata[1]}<br>x=%{x}<br>y=%{y}<extra>'+ex+'</extra>'),row=1,col=col)
        fig.update_xaxes(title_text=symbol(FEATURE),row=1,col=col)
fig.update_yaxes(title_text='hours',row=1,col=1);fig.update_yaxes(title_text='%',row=1,col=2)
fig.update_layout(template='plotly_white',height=480);fig.show()

## 5. Prediction: hold out entire examples

Each fold holds out all children of one example. Training-only median imputation, constant removal and standardization precede fixed Ridge regression (alpha=10). Training examples have equal total weight. Zero change and the training-example-weighted mean are explicit baselines. No feature selection or hyperparameter search uses the held-out examples.

Results report both pair-weighted and equal-example errors. Prefer the latter for transfer: a large network should not dominate just because it supplies more collapses. R² and MAE answer different questions; a model can improve squared error while losing to zero change on typical absolute error.

`partial` is the main cohort. `including_centerlines` adds the 15 global collapses and tests their leverage. Every cohort is refitted with 54 excluded from both training and evaluation. `network_on_domain` restricts the network panels to exactly the available local-domain records for fair scope comparisons. Local partial-collapse validation covers only the examples with validated local records; missing examples are not claimed as tested.

These panels were fixed before viewing current scores. Further decisions based on these scores make this cohort exploratory; subsequent variable selection needs nested example-level validation. Geometric lengths alone provide no physical velocity scale, so predicting hours remains conditional on the event/routing regimes represented here.

In [18]:
COHORT='partial'  # also 'including_centerlines'
view=scores[scores.weighting.eq('equal_example')&scores.run.str.startswith(COHORT+'/')&scores.response.isin(['peak_shift_hours','peak_excess_change_pct'])].copy()
view['mae_minutes_for_timing']=np.where(view.response.eq('peak_shift_hours'),60*view.mae,np.nan)
show_symbols(view[['run','panel','response','n','examples','mae','mae_minutes_for_timing','rmse','r2']].round(4))
# All six targets remain available; avoid choosing a different target to rescue a weak primary result.
show_symbols(scores[scores.weighting.eq('equal_example')&scores.run.eq(COHORT+'/domain/all')][['panel','response','mae','r2']].round(4))

|  | run | panel | response | n | examples | mae | mae_minutes_for_timing | rmse | r2 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 3016 | partial/domain/all | length_exposure | $100(R_{E_{\max}}-1)\;[\%]$ | 1569 | 13 | 0.0616 | nan | 0.1303 | 0.1629 |
| 3031 | partial/domain/all | length_exposure | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1569 | 13 | 0.1147 | 6.8803 | 0.1961 | 0.459 |
| 3106 | partial/domain/all | length_width_exposure | $100(R_{E_{\max}}-1)\;[\%]$ | 1569 | 13 | 0.0603 | nan | 0.1297 | 0.1709 |
| 3121 | partial/domain/all | length_width_exposure | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1569 | 13 | 0.104 | 6.2384 | 0.1891 | 0.4973 |
| 3196 | partial/domain/all | length_width_topology_exposure | $100(R_{E_{\max}}-1)\;[\%]$ | 1569 | 13 | 0.0614 | nan | 0.1319 | 0.1432 |
| 3211 | partial/domain/all | length_width_topology_exposure | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1569 | 13 | 0.1039 | 6.2312 | 0.1949 | 0.466 |
| 3286 | partial/domain/all | training_mean | $100(R_{E_{\max}}-1)\;[\%]$ | 1569 | 13 | 0.0655 | nan | 0.1436 | -0.0163 |
| 3301 | partial/domain/all | training_mean | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1569 | 13 | 0.1391 | 8.3452 | 0.2696 | -0.0222 |
| 3376 | partial/domain/all | zero | $100(R_{E_{\max}}-1)\;[\%]$ | 1569 | 13 | 0.0554 | nan | 0.1487 | -0.0893 |
| 3391 | partial/domain/all | zero | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1569 | 13 | 0.137 | 8.2204 | 0.296 | -0.2316 |
| 3463 | partial/domain/without54 | length_exposure | $100(R_{E_{\max}}-1)\;[\%]$ | 1493 | 12 | 0.0656 | nan | 0.1358 | 0.1495 |
| 3477 | partial/domain/without54 | length_exposure | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1493 | 12 | 0.1219 | 7.3153 | 0.2079 | 0.4307 |
| 3547 | partial/domain/without54 | length_width_exposure | $100(R_{E_{\max}}-1)\;[\%]$ | 1493 | 12 | 0.0633 | nan | 0.1343 | 0.1671 |
| 3561 | partial/domain/without54 | length_width_exposure | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1493 | 12 | 0.1083 | 6.4969 | 0.1949 | 0.4998 |
| 3631 | partial/domain/without54 | length_width_topology_exposure | $100(R_{E_{\max}}-1)\;[\%]$ | 1493 | 12 | 0.0652 | nan | 0.1373 | 0.1303 |
| 3645 | partial/domain/without54 | length_width_topology_exposure | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1493 | 12 | 0.1103 | 6.6202 | 0.2032 | 0.4561 |
| 3715 | partial/domain/without54 | training_mean | $100(R_{E_{\max}}-1)\;[\%]$ | 1493 | 12 | 0.0682 | nan | 0.1485 | -0.0175 |
| 3729 | partial/domain/without54 | training_mean | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1493 | 12 | 0.1447 | 8.6847 | 0.2787 | -0.0231 |
| 3799 | partial/domain/without54 | zero | $100(R_{E_{\max}}-1)\;[\%]$ | 1493 | 12 | 0.0587 | nan | 0.1539 | -0.0933 |
| 3813 | partial/domain/without54 | zero | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1493 | 12 | 0.1446 | 8.6751 | 0.3069 | -0.2408 |
| 3892 | partial/network/all | length | $100(R_{E_{\max}}-1)\;[\%]$ | 2172 | 15 | 0.0613 | nan | 0.1277 | 0.175 |
| 3909 | partial/network/all | length | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 2172 | 15 | 0.1127 | 6.7619 | 0.1912 | 0.4665 |
| 3994 | partial/network/all | length_width | $100(R_{E_{\max}}-1)\;[\%]$ | 2172 | 15 | 0.0601 | nan | 0.1259 | 0.1985 |
| 4011 | partial/network/all | length_width | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 2172 | 15 | 0.1151 | 6.9034 | 0.2449 | 0.1241 |
| 4096 | partial/network/all | length_width_topology | $100(R_{E_{\max}}-1)\;[\%]$ | 2172 | 15 | 0.0602 | nan | 0.1276 | 0.1769 |
| 4113 | partial/network/all | length_width_topology | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 2172 | 15 | 0.1151 | 6.905 | 0.2296 | 0.2304 |
| 4198 | partial/network/all | training_mean | $100(R_{E_{\max}}-1)\;[\%]$ | 2172 | 15 | 0.065 | nan | 0.1416 | -0.0143 |
| 4215 | partial/network/all | training_mean | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 2172 | 15 | 0.1401 | 8.4063 | 0.2645 | -0.0215 |
| 4300 | partial/network/all | zero | $100(R_{E_{\max}}-1)\;[\%]$ | 2172 | 15 | 0.0561 | nan | 0.1457 | -0.0731 |
| 4317 | partial/network/all | zero | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 2172 | 15 | 0.1382 | 8.2943 | 0.2884 | -0.2146 |
| 4399 | partial/network/without54 | length | $100(R_{E_{\max}}-1)\;[\%]$ | 2091 | 14 | 0.0637 | nan | 0.1316 | 0.1719 |
| 4415 | partial/network/without54 | length | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 2091 | 14 | 0.1164 | 6.9821 | 0.1963 | 0.4685 |
| 4495 | partial/network/without54 | length_width | $100(R_{E_{\max}}-1)\;[\%]$ | 2091 | 14 | 0.0626 | nan | 0.1306 | 0.1834 |
| 4511 | partial/network/without54 | length_width | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 2091 | 14 | 0.1181 | 7.0834 | 0.2394 | 0.2089 |
| 4591 | partial/network/without54 | length_width_topology | $100(R_{E_{\max}}-1)\;[\%]$ | 2091 | 14 | 0.0626 | nan | 0.1336 | 0.1462 |
| 4607 | partial/network/without54 | length_width_topology | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 2091 | 14 | 0.1178 | 7.0705 | 0.2255 | 0.2986 |
| 4687 | partial/network/without54 | training_mean | $100(R_{E_{\max}}-1)\;[\%]$ | 2091 | 14 | 0.067 | nan | 0.1457 | -0.0155 |
| 4703 | partial/network/without54 | training_mean | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 2091 | 14 | 0.1449 | 8.6946 | 0.2722 | -0.0226 |
| 4783 | partial/network/without54 | zero | $100(R_{E_{\max}}-1)\;[\%]$ | 2091 | 14 | 0.0587 | nan | 0.1499 | -0.0744 |
| 4799 | partial/network/without54 | zero | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 2091 | 14 | 0.1445 | 8.6698 | 0.2974 | -0.2201 |
| 4876 | partial/network_on_domain/all | length | $100(R_{E_{\max}}-1)\;[\%]$ | 1569 | 13 | 0.062 | nan | 0.1387 | 0.0515 |
| 4891 | partial/network_on_domain/all | length | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1569 | 13 | 0.1062 | 6.3741 | 0.2026 | 0.4226 |
| 4966 | partial/network_on_domain/all | length_width | $100(R_{E_{\max}}-1)\;[\%]$ | 1569 | 13 | 0.0594 | nan | 0.1349 | 0.1036 |
| 4981 | partial/network_on_domain/all | length_width | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1569 | 13 | 0.1076 | 6.4548 | 0.2613 | 0.0396 |
| 5056 | partial/network_on_domain/all | length_width_topology | $100(R_{E_{\max}}-1)\;[\%]$ | 1569 | 13 | 0.0585 | nan | 0.1442 | -0.0247 |
| 5071 | partial/network_on_domain/all | length_width_topology | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1569 | 13 | 0.1034 | 6.2012 | 0.2162 | 0.343 |
| 5146 | partial/network_on_domain/all | training_mean | $100(R_{E_{\max}}-1)\;[\%]$ | 1569 | 13 | 0.0655 | nan | 0.1436 | -0.0163 |
| 5161 | partial/network_on_domain/all | training_mean | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1569 | 13 | 0.1391 | 8.3452 | 0.2696 | -0.0222 |
| 5236 | partial/network_on_domain/all | zero | $100(R_{E_{\max}}-1)\;[\%]$ | 1569 | 13 | 0.0554 | nan | 0.1487 | -0.0893 |
| 5251 | partial/network_on_domain/all | zero | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1569 | 13 | 0.137 | 8.2204 | 0.296 | -0.2316 |
| 5323 | partial/network_on_domain/without54 | length | $100(R_{E_{\max}}-1)\;[\%]$ | 1493 | 12 | 0.0654 | nan | 0.1448 | 0.0323 |
| 5337 | partial/network_on_domain/without54 | length | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1493 | 12 | 0.11 | 6.6015 | 0.2174 | 0.3775 |
| 5407 | partial/network_on_domain/without54 | length_width | $100(R_{E_{\max}}-1)\;[\%]$ | 1493 | 12 | 0.064 | nan | 0.1539 | -0.0937 |
| 5421 | partial/network_on_domain/without54 | length_width | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1493 | 12 | 0.1124 | 6.7418 | 0.2216 | 0.3532 |
| 5491 | partial/network_on_domain/without54 | length_width_topology | $100(R_{E_{\max}}-1)\;[\%]$ | 1493 | 12 | 0.0631 | nan | 0.1769 | -0.4438 |
| 5505 | partial/network_on_domain/without54 | length_width_topology | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1493 | 12 | 0.103 | 6.1822 | 0.1923 | 0.513 |
| 5575 | partial/network_on_domain/without54 | training_mean | $100(R_{E_{\max}}-1)\;[\%]$ | 1493 | 12 | 0.0682 | nan | 0.1485 | -0.0175 |
| 5589 | partial/network_on_domain/without54 | training_mean | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1493 | 12 | 0.1447 | 8.6847 | 0.2787 | -0.0231 |
| 5659 | partial/network_on_domain/without54 | zero | $100(R_{E_{\max}}-1)\;[\%]$ | 1493 | 12 | 0.0587 | nan | 0.1539 | -0.0933 |
| 5673 | partial/network_on_domain/without54 | zero | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 1493 | 12 | 0.1446 | 8.6751 | 0.3069 | -0.2408 |

|  | panel | response | mae | r2 |
| --- | --- | --- | --- | --- |
| 2971 | length_exposure | $\Delta A$ | 0.0004 | 0.3623 |
| 2986 | length_exposure | $\Delta\mu_t\;[\mathrm{h}]$ | 0.0774 | 0.5445 |
| 3001 | length_exposure | $100(R_{D_{80}}-1)\;[\%]$ | 0.0698 | -0.0223 |
| 3016 | length_exposure | $100(R_{E_{\max}}-1)\;[\%]$ | 0.0616 | 0.1629 |
| 3031 | length_exposure | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 0.1147 | 0.459 |
| 3046 | length_exposure | $100(R_V-1)\;[\%]$ | 0.041 | 0.0178 |
| 3061 | length_width_exposure | $\Delta A$ | 0.0004 | 0.3435 |
| 3076 | length_width_exposure | $\Delta\mu_t\;[\mathrm{h}]$ | 0.0714 | 0.5844 |
| 3091 | length_width_exposure | $100(R_{D_{80}}-1)\;[\%]$ | 0.0686 | -0.0473 |
| 3106 | length_width_exposure | $100(R_{E_{\max}}-1)\;[\%]$ | 0.0603 | 0.1709 |
| 3121 | length_width_exposure | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 0.104 | 0.4973 |
| 3136 | length_width_exposure | $100(R_V-1)\;[\%]$ | 0.0405 | -0.0178 |
| 3151 | length_width_topology_exposure | $\Delta A$ | 0.0004 | 0.3115 |
| 3166 | length_width_topology_exposure | $\Delta\mu_t\;[\mathrm{h}]$ | 0.071 | 0.5451 |
| 3181 | length_width_topology_exposure | $100(R_{D_{80}}-1)\;[\%]$ | 0.0732 | -0.2323 |
| 3196 | length_width_topology_exposure | $100(R_{E_{\max}}-1)\;[\%]$ | 0.0614 | 0.1432 |
| 3211 | length_width_topology_exposure | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 0.1039 | 0.466 |
| 3226 | length_width_topology_exposure | $100(R_V-1)\;[\%]$ | 0.0427 | -0.2568 |
| 3241 | training_mean | $\Delta A$ | 0.0004 | -0.0085 |
| 3256 | training_mean | $\Delta\mu_t\;[\mathrm{h}]$ | 0.1011 | -0.0169 |
| 3271 | training_mean | $100(R_{D_{80}}-1)\;[\%]$ | 0.0587 | -0.0142 |
| 3286 | training_mean | $100(R_{E_{\max}}-1)\;[\%]$ | 0.0655 | -0.0163 |
| 3301 | training_mean | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 0.1391 | -0.0222 |
| 3316 | training_mean | $100(R_V-1)\;[\%]$ | 0.0409 | -0.0267 |
| 3331 | zero | $\Delta A$ | 0.0004 | -0.062 |
| 3346 | zero | $\Delta\mu_t\;[\mathrm{h}]$ | 0.1051 | -0.2462 |
| 3361 | zero | $100(R_{D_{80}}-1)\;[\%]$ | 0.0554 | -0.0001 |
| 3376 | zero | $100(R_{E_{\max}}-1)\;[\%]$ | 0.0554 | -0.0893 |
| 3391 | zero | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 0.137 | -0.2316 |
| 3406 | zero | $100(R_V-1)\;[\%]$ | 0.0356 | -0.0145 |

In [19]:
RUN='partial/domain/all'
PANEL='length_width_exposure'
TARGET='peak_shift_hours'
pred=pd.read_parquet(OUT/'held_example_predictions.parquet')
v=pred[(pred.run==RUN)&(pred.panel==PANEL)&(pred.response==TARGET)]
fig=px.scatter(v,x='observed',y='predicted',color='example',hover_data=['state'],
    title='Unseen-example predictions — '+METRICS[TARGET][0])
lo=min(v.observed.min(),v.predicted.min());hi=max(v.observed.max(),v.predicted.max())
fig.add_shape(type='line',x0=lo,x1=hi,y0=lo,y1=hi,line=dict(dash='dash',color='grey'))
fig.update_layout(template='plotly_white',height=520,xaxis_title='Observed '+symbol(TARGET),yaxis_title='Predicted '+symbol(TARGET));fig.show()
show_symbols(scores[(scores.run==RUN)&(scores.panel==PANEL)&(scores.response==TARGET)&scores.weighting.str.startswith('example_')][['weighting','n','mae','rmse','r2']].round(4))

|  | weighting | n | mae | rmse | r2 |
| --- | --- | --- | --- | --- | --- |
| 3122 | example_02 | 29 | 0.0744 | 0.1095 | 0.6944 |
| 3123 | example_03 | 36 | 0.1095 | 0.1456 | -0.5663 |
| 3124 | example_05 | 50 | 0.2403 | 0.3942 | -0.1837 |
| 3125 | example_21 | 26 | 0.1038 | 0.1196 | -3.0816 |
| 3126 | example_22 | 329 | 0.0445 | 0.0902 | 0.6651 |
| 3127 | example_37 | 320 | 0.039 | 0.0924 | 0.9668 |
| 3128 | example_41 | 119 | 0.1144 | 0.259 | 0.4452 |
| 3129 | example_45 | 93 | 0.0883 | 0.1506 | 0.3363 |
| 3130 | example_46 | 207 | 0.1218 | 0.2424 | 0.2951 |
| 3131 | example_48 | 71 | 0.124 | 0.1671 | -37.3169 |
| 3132 | example_54 | 76 | 0.1299 | 0.1623 | -3.4032 |
| 3133 | example_55 | 196 | 0.0793 | 0.1763 | 0.2981 |
| 3134 | example_56 | 17 | 0.0826 | 0.1062 | 0.765 |

## 6. Example 54: global maxima and a separate diagnostic

The parent has two close-in-height crests around 13:43:30 and 22:45:30 UTC on 22 June 2023, approximately 1,393.39 and 1,396.39 m³/s. The small dip after the first is not evidence that it is numerically absent; a large prominence cutoff would erase it.

**Provisional diagnostic:** find local maxima separated by at least two hours and within 1% of the hydrograph's maximum excess. Match each of the two parent crests to exactly one candidate within a specified time radius, require chronological order, and require the child's global maximum to belong to the matched pair. Compare which ordered crest is dominant. No prominence cutoff is used. Default matching radius: 4.5 h; sensitivity: 2 h and 6 h.

The flag is three-state: resolved switch, resolved no switch, or unresolved. It is a heuristic based on temporal correspondence, not proof that a water parcel or the same physical wave was tracked. A child-derived flag is **diagnostic only**, never a predictor. The 1% here is a crest-height diagnostic tolerance, unrelated to the analysis-window tolerance.

All 81 partial-collapse children resolve as no switch under all three radii. The single centerline is unresolved at 2 and 4.5 h but classified as a switch at 6 h. Its large peak shift therefore needs qualified interpretation, not a forced binary label. Neither this flag nor matched-crest timing replaces any of the six response values.

54 ends above its baseline. Centroid, duration, asymmetry and volume describe the finite-window excess including that transition, not an isolated pulse returning to its starting discharge.

In [20]:
crest=pd.read_csv(OUT/'crest_diagnostic_54.csv',dtype={'example':str})
show_symbols(pd.read_csv(OUT/'parent_crests_54.csv'))
show_symbols(crest.groupby(['matching_radius_hours','status','crest_switch_flag'],dropna=False).size().rename('comparisons'))
show_symbols(crest[crest.state.eq('fully_collapsed')])
show_symbols(d[d.example.eq('54')][['state',*METRICS]].sort_values('peak_shift_hours').head(8))
threshold_figure(states,'54','fully_collapsed',view='peak',gauges=False).show()

|  | crest | time_utc | peak_cms |
| --- | --- | --- | --- |
| 0 | 1 | 2023-06-22 13:43:30+00:00 | 1393.385928532223 |
| 1 | 2 | 2023-06-22 22:45:30+00:00 | 1396.390721972938 |

|  | comparisons |
| --- | --- |
| (2.0, 'resolved_two_crests', False) | 81 |
| (2.0, 'unresolved', nan) | 1 |
| (4.5, 'resolved_two_crests', False) | 81 |
| (4.5, 'unresolved', nan) | 1 |
| (6.0, 'resolved_two_crests', False) | 81 |
| (6.0, 'resolved_two_crests', True) | 1 |

|  | example | state | matching_radius_hours | status | crest_switch_flag | dominant_crest | left_peak_seconds | right_peak_seconds | left_peak_cms | right_peak_cms |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 243 | 54 | fully_collapsed | 2.0 | unresolved | nan | nan | nan | nan | nan | nan |
| 244 | 54 | fully_collapsed | 4.5 | unresolved | nan | nan | nan | nan | nan | nan |
| 245 | 54 | fully_collapsed | 6.0 | resolved_two_crests | True | 1.0 | 1687421790.0 | 1687470330.0 | 1422.8211147150907 | 1412.8025047732394 |

|  | state | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | $100(R_{E_{\max}}-1)\;[\%]$ | $\Delta\mu_t\;[\mathrm{h}]$ | $100(R_{D_{80}}-1)\;[\%]$ | $\Delta A$ | $100(R_V-1)\;[\%]$ |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1947 | fully_collapsed | -14.483333333333333 | 2.366426212103523 | -2.948885714676645 | 2.830592943031296 | 0.022238673204397028 | 2.694976958752693 |
| 1873 | CB004_bubble_16 | -0.45 | 0.2531923673421588 | -0.38623379468917846 | 0.3105570070528918 | 0.003468392996037087 | 0.35424282061189416 |
| 1949 | island_54_033 | -0.38333333333333336 | 0.21713160209113624 | -0.07342561503251394 | 0.0029466318279824932 | 0.0011558774239334296 | 0.09216121805535327 |
| 1872 | CB003_bubble_12 | -0.35 | 0.016093209544276732 | -0.28142136368486614 | 0.31049029042631293 | 0.0015340875043306362 | 0.2293218384779916 |
| 1946 | S066_unit_4 | -0.2833333333333333 | 0.2465674905477 | -0.24979384104410807 | 0.1605776302175732 | 0.0027122352073134792 | 0.24316602451173175 |
| 1945 | S065_unit_9 | -0.21666666666666667 | 0.2399759739210916 | -0.16001094586319395 | 0.06441606077662865 | 0.0021724852900415204 | 0.16961588082977652 |
| 1871 | CB002_bubble_11 | -0.18333333333333332 | 0.0052520100135344805 | -0.130054609047042 | 0.1445634486983094 | 0.0006899695682702905 | 0.10579608910832317 |
| 1948 | island_54_009 | -0.18333333333333332 | 0.19434160767248798 | -0.04689846012327406 | -0.012111850902152277 | 0.0008352239330186584 | 0.06585140732813244 |

## 7. Window sensitivity

The baseline is held fixed. Both global-peak targets are identical across the three finalized windows. The four distribution responses can change; inspect maximum as well as median differences, because a small median can conceal an influential centerline or equilibrium transition.

`sensitivity_validation_scores.csv` evaluates the same held-example predictions trained on 0.5% responses against 1% and 2% labels. It is a label-sensitivity check, not a separately tuned model.

In [21]:
sensitivity=pd.read_csv(OUT/'window_sensitivity.csv')
show_symbols(sensitivity[sensitivity.scope.eq('all')].round(6))
show_symbols(sensitivity[sensitivity.scope.eq('54')].round(6))

|  | window | response | scope | n | median_absolute_change | max_absolute_change | sign_reversals |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | 1% | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | all | 2187 | 0.0 | 0.0 | 0 |
| 3 | 1% | $100(R_{E_{\max}}-1)\;[\%]$ | all | 2187 | 0.0 | 0.0 | 0 |
| 6 | 1% | $\Delta\mu_t\;[\mathrm{h}]$ | all | 2187 | 0.000145 | 2.330002 | 4 |
| 9 | 1% | $100(R_{D_{80}}-1)\;[\%]$ | all | 2187 | 0.00045 | 1.042463 | 10 |
| 12 | 1% | $\Delta A$ | all | 2187 | 3e-06 | 0.012488 | 11 |
| 15 | 1% | $100(R_V-1)\;[\%]$ | all | 2187 | 0.000191 | 0.743779 | 15 |
| 18 | 2% | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | all | 2187 | 0.0 | 0.0 | 0 |
| 21 | 2% | $100(R_{E_{\max}}-1)\;[\%]$ | all | 2187 | 0.0 | 0.0 | 0 |
| 24 | 2% | $\Delta\mu_t\;[\mathrm{h}]$ | all | 2187 | 0.000398 | 4.743034 | 15 |
| 27 | 2% | $100(R_{D_{80}}-1)\;[\%]$ | all | 2187 | 0.001117 | 2.137799 | 25 |
| 30 | 2% | $\Delta A$ | all | 2187 | 7e-06 | 0.029044 | 26 |
| 33 | 2% | $100(R_V-1)\;[\%]$ | all | 2187 | 0.000563 | 1.828451 | 26 |

|  | window | response | scope | n | median_absolute_change | max_absolute_change | sign_reversals |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 1% | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 54 | 82 | 0.0 | 0.0 | 0 |
| 4 | 1% | $100(R_{E_{\max}}-1)\;[\%]$ | 54 | 82 | 0.0 | 0.0 | 0 |
| 7 | 1% | $\Delta\mu_t\;[\mathrm{h}]$ | 54 | 82 | 5e-06 | 0.003788 | 0 |
| 10 | 1% | $100(R_{D_{80}}-1)\;[\%]$ | 54 | 82 | 0.000459 | 0.033321 | 0 |
| 13 | 1% | $\Delta A$ | 54 | 82 | 8e-06 | 0.000573 | 0 |
| 16 | 1% | $100(R_V-1)\;[\%]$ | 54 | 82 | 0.000689 | 0.087925 | 0 |
| 19 | 2% | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | 54 | 82 | 0.0 | 0.0 | 0 |
| 22 | 2% | $100(R_{E_{\max}}-1)\;[\%]$ | 54 | 82 | 0.0 | 0.0 | 0 |
| 25 | 2% | $\Delta\mu_t\;[\mathrm{h}]$ | 54 | 82 | 0.000217 | 0.044087 | 0 |
| 28 | 2% | $100(R_{D_{80}}-1)\;[\%]$ | 54 | 82 | 0.000878 | 0.044939 | 1 |
| 31 | 2% | $\Delta A$ | 54 | 82 | 2.1e-05 | 0.00149 | 0 |
| 34 | 2% | $100(R_V-1)\;[\%]$ | 54 | 82 | 0.00169 | 0.211582 | 0 |

## 8. What this pass supports—and what comes next

1. Review peak timing first, then peak excess, using the same geometry panels. Use centroid/duration/asymmetry to interpret cases where peak timing alone is misleading; none is an input to the geometric predictor.
2. Resolve domain coverage before final local-variable selection: missing membership, changed endpoints and anchor correspondence are different problems. Examples 13 and 15 currently have no validated partial-collapse domains. The 1,078 containing-compound contexts must remain distinct from the 491 closed-unit domains.
3. Rebuild equivalent resistance under the explicit width–depth laws on validated domains, then compare its compact panel on the same cohort. Do not substitute historical resistance values or calibrated hydraulic timing.
4. Compare alternative representations within length, width balance and topology, with selection inside training examples. Keep a variable for a clear mechanism and repeatable incremental value, not just a pooled correlation. Investigate residual patterns without adding child-derived diagnostics as predictors.
5. Examine weak amplitude predictions explicitly. A small positive R² is not sufficient if zero change has a lower MAE. Sparse near-zero responses and a few large cases can reward different loss functions.

Scripts in `RAPID/investigations/hydrograph_predictors_20261014/` reproduce extraction, audits and validation. Inputs and definitions are hashed in the output manifest. Existing response and worked-example notebooks are preserved.

## 9. Read the cohort comparison before quoting a score

For partial collapses, the whole-network length panel covers all 15 examples (2,172 comparisons). The local length/width/exposure panel covers 13 examples (1,569 comparisons); these two scores are not a head-to-head comparison. Use `network_on_domain` for that comparison.

The local panel's partial-collapse timing R² is approximately 0.50, with MAE about 6.2 minutes. Excluding 54 leaves R² approximately 0.50. This is a limited-cohort benchmark, not evidence for a universal predictor.

**Including single centerlines causes the local panel to fail badly** (negative R²), even after excluding 54. The response scale and geometric changes are much larger, and incomplete local coverage gives examples 13 and 15 only one available observation each—the centerline—which receives an entire example's training weight. Inspect this coverage and leverage before drawing conclusions about global-collapse physics. A simple pooled linear local model is not justified across both regimes by these results.

The network panel remains available on all records, with full-cohort errors reported below. Peak-excess models have positive R² in some panels but worse MAE than zero change: predictive value is not established uniformly across losses.

In [22]:
summary=scores[scores.weighting.eq('equal_example')&scores.response.eq('peak_shift_hours')&(
    (scores.run.str.contains('/domain/')&scores.panel.eq('length_width_exposure'))|
    (scores.run.str.contains('/network/')&scores.panel.eq('length')))].copy()
summary['mae_minutes']=60*summary.mae
show_symbols(summary[['run','panel','n','examples','mae_minutes','r2']].round(4))

|  | run | panel | n | examples | mae_minutes | r2 |
| --- | --- | --- | --- | --- | --- | --- |
| 171 | including_centerlines/domain/all | length_width_exposure | 1584 | 15 | 314.0513 | -3.5073 |
| 671 | including_centerlines/domain/without54 | length_width_exposure | 1507 | 14 | 336.781 | -3.6939 |
| 1059 | including_centerlines/network/all | length | 2187 | 15 | 15.0406 | 0.3265 |
| 1565 | including_centerlines/network/without54 | length | 2105 | 14 | 14.9804 | 0.2904 |
| 3121 | partial/domain/all | length_width_exposure | 1569 | 13 | 6.2384 | 0.4973 |
| 3561 | partial/domain/without54 | length_width_exposure | 1493 | 12 | 6.4969 | 0.4998 |
| 3909 | partial/network/all | length | 2172 | 15 | 6.7619 | 0.4665 |
| 4415 | partial/network/without54 | length | 2091 | 14 | 6.9821 | 0.4685 |

## Code-name lookup

Implementation names are retained here for reproducibility; labels above use symbols.

In [23]:
keys=list(dict.fromkeys([f for key in ['network_panels','domain_panels'] for panel in design[key].values() for f in panel]+list(RESPONSE_SYMBOLS)))
lookup=pd.DataFrame({'Symbol':[symbol(k) for k in keys],'Code name':['`'+k+'`' for k in keys]})
show_symbols(lookup)

|  | Symbol | Code name |
| --- | --- | --- |
| 0 | $R_{\mu_L}^{N}$ | `network_ratio_route_length_mean` |
| 1 | $(\mu_L)_p^{N}$ | `network_parent_route_length_mean` |
| 2 | $\Delta(\mathrm{CV}_L)^{N}$ | `network_delta_route_length_cv` |
| 3 | $R_{\bar W}^{N}$ | `network_ratio_width_mean` |
| 4 | $\Delta(\mathrm{CV}_W)^{N}$ | `network_delta_width_cv` |
| 5 | $\Delta(\mathbb{E}[N_b])^{N}$ | `network_delta_expected_splits` |
| 6 | $r_{\mathrm{in},p}^{D}\,\Delta\mu_L^{D}/\mu_{L,p}^{N}$ | `exposed_length_change` |
| 7 | $\mu_{L,p}^{D}/\mu_{L,p}^{N}$ | `domain_parent_length_over_network` |
| 8 | $r_{\mathrm{in},p}^{D}\,\Delta\sigma_L^{D}/\mu_{L,p}^{N}$ | `exposed_route_sd_change` |
| 9 | $r_{\mathrm{in},p}^{D}$ | `domain_entry_exposure` |
| 10 | $R_{\bar W}^{D}$ | `domain_ratio_width_mean` |
| 11 | $(\bar d)_p^{D}$ | `domain_parent_dominance_mean` |
| 12 | $\Delta(\mathbb{E}[N_b])^{D}$ | `domain_delta_expected_splits` |
| 13 | $\Delta t_{\mathrm{peak}}\;[\mathrm{h}]$ | `peak_shift_hours` |
| 14 | $100(R_{E_{\max}}-1)\;[\%]$ | `peak_excess_change_pct` |
| 15 | $\Delta\mu_t\;[\mathrm{h}]$ | `centroid_shift_hours` |
| 16 | $100(R_{D_{80}}-1)\;[\%]$ | `duration80_change_pct` |
| 17 | $100(R_V-1)\;[\%]$ | `volume_change_pct` |
| 18 | $\Delta A$ | `asymmetry_change` |